<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>Claude API for Python Developers</h1>
<h1>5. Code Generation and Explanation</h1>
        <p>Bruno Gonçalves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In [1]:
import anthropic
from IPython.display import display, Markdown, Code

import matplotlib.pyplot as plt

import watermark

%load_ext watermark
%matplotlib inline

We start by print out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.13.3
IPython version      : 9.8.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 25.1.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: b09c334a256fa7d396f7c25e28a2e0ab2f1a3604

matplotlib: 3.10.7
anthropic : 0.75.0
watermark : 2.5.0
IPython   : 9.8.0



Load default figure style

In [3]:
plt.style.use('d4sci.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

## Setup and Imports


In [4]:

# Initialize the client
client = anthropic.Anthropic()

def show_code(code, language="python"):
    """Display code with syntax highlighting."""
    display(Markdown(f"```{language}\n{code}\n```"))

print("Client initialized successfully!")


Client initialized successfully!


---
## 1. Generating Code from Prompts

Claude excels at generating code from natural language descriptions. The key is providing clear, specific requirements.


In [5]:
# Basic code generation
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    system="You are an expert Python developer. Write clean, well-documented code.",
    messages=[
        {
            "role": "user",
            "content": "Write a Python function that checks if a string is a valid email address using regex."
        }
    ]
)

print("📝 Generated Code:")
display(Markdown(response.content[0].text))


📝 Generated Code:


Here's a Python function that validates email addresses using regex:

```python
import re

def is_valid_email(email):
    """
    Check if a string is a valid email address using regex.
    
    This function validates email addresses based on RFC 5322 standards with some
    practical limitations for common use cases.
    
    Args:
        email (str): The email address string to validate
        
    Returns:
        bool: True if the email is valid, False otherwise
        
    Examples:
        >>> is_valid_email("user@example.com")
        True
        >>> is_valid_email("invalid.email")
        False
        >>> is_valid_email("user.name+tag@example-site.co.uk")
        True
    """
    if not isinstance(email, str):
        return False
    
    # Comprehensive email regex pattern
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    
    # Additional validation for edge cases
    # Check for consecutive dots
    if '..' in email:
        return False
    
    # Check if email starts or ends with a dot
    local_part = email.split('@')[0] if '@' in email else email
    if local_part.startswith('.') or local_part.endswith('.'):
        return False
    
    return re.match(pattern, email) is not None


def is_valid_email_strict(email):
    """
    More comprehensive email validation with stricter rules.
    
    This function provides additional validation beyond basic regex matching,
    including length limits and more detailed format checking.
    
    Args:
        email (str): The email address string to validate
        
    Returns:
        bool: True if the email is valid, False otherwise
        
    Examples:
        >>> is_valid_email_strict("test@example.com")
        True
        >>> is_valid_email_strict("a" * 65 + "@example.com")
        False
    """
    if not isinstance(email, str) or len(email) > 254:
        return False
    
    # Split email into local and domain parts
    parts = email.split('@')
    if len(parts) != 2:
        return False
    
    local_part, domain_part = parts
    
    # Validate local part (before @)
    if not local_part or len(local_part) > 64:
        return False
    
    # Validate domain part (after @)
    if not domain_part or len(domain_part) > 253:
        return False
    
    # Regex patterns for local and domain parts
    local_pattern = r'^[a-zA-Z0-9._%+-]+$'
    domain_pattern = r'^[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    
    # Additional checks
    if (local_part.startswith('.') or local_part.endswith('.') or 
        '..' in local_part or '..' in domain_part):
        return False
    
    # Validate using regex
    return (re.match(local_pattern, local_part) is not None and 
            re.match(domain_pattern, domain_part) is not None)


# Test function to demonstrate usage
def test_email_validation():
    """Test function to demonstrate email validation with various examples."""
    test_emails = [
        ("user@example.com", True),
        ("test.email@example.org", True),
        ("user+tag@example-site.co.uk", True),
        ("firstname.lastname@domain.com", True),
        ("email@123.123.123.123", False),  # IP addresses not supported in this regex
        ("plainaddress", False),
        ("@missinglocalpart.com", False),
        ("missing@.com", False),
        ("missing@domain", False),
        ("spaces @example.com", False),
        ("user@", False),
        ("user@domain..com", False),
        (".user@domain.com", False),
        ("user.@domain.com", False),
        ("user..name@domain.com", False),

In [6]:
# More specific requirements lead to better code
detailed_prompt = """
Write a Python class called `RateLimiter` with the following specifications:

1. Constructor takes `max_requests` (int) and `time_window` (int, seconds)
2. Method `is_allowed()` returns True if request is within rate limit
3. Method `reset()` clears the request history
4. Use a sliding window algorithm
5. Include type hints
6. Add docstrings

Example usage:
```python
limiter = RateLimiter(max_requests=5, time_window=60)
if limiter.is_allowed():
    # Process request
```
"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    system="You are an expert Python developer. Write production-quality code.",
    messages=[{"role": "user", "content": detailed_prompt}]
)

print("📝 Generated Rate Limiter:")
display(Markdown(response.content[0].text))


📝 Generated Rate Limiter:


```python
import time
from collections import deque
from typing import Deque


class RateLimiter:
    """
    A rate limiter implementation using a sliding window algorithm.
    
    This class tracks request timestamps and allows requests only if they
    fall within the specified rate limit over a sliding time window.
    
    Attributes:
        max_requests (int): Maximum number of requests allowed in the time window
        time_window (int): Time window in seconds
    """
    
    def __init__(self, max_requests: int, time_window: int) -> None:
        """
        Initialize the RateLimiter.
        
        Args:
            max_requests (int): Maximum number of requests allowed in the time window.
                              Must be a positive integer.
            time_window (int): Time window in seconds. Must be a positive integer.
            
        Raises:
            ValueError: If max_requests or time_window is not a positive integer.
        """
        if not isinstance(max_requests, int) or max_requests <= 0:
            raise ValueError("max_requests must be a positive integer")
        if not isinstance(time_window, int) or time_window <= 0:
            raise ValueError("time_window must be a positive integer")
            
        self.max_requests = max_requests
        self.time_window = time_window
        self._request_timestamps: Deque[float] = deque()
    
    def is_allowed(self) -> bool:
        """
        Check if a request is allowed based on the current rate limit.
        
        This method implements a sliding window algorithm by:
        1. Removing timestamps older than the time window
        2. Checking if adding a new request would exceed the limit
        3. If allowed, recording the current timestamp
        
        Returns:
            bool: True if the request is allowed, False otherwise.
        """
        current_time = time.time()
        
        # Remove timestamps outside the current time window
        cutoff_time = current_time - self.time_window
        while self._request_timestamps and self._request_timestamps[0] <= cutoff_time:
            self._request_timestamps.popleft()
        
        # Check if we can allow this request
        if len(self._request_timestamps) < self.max_requests:
            self._request_timestamps.append(current_time)
            return True
        
        return False
    
    def reset(self) -> None:
        """
        Clear the request history.
        
        This removes all recorded request timestamps, effectively resetting
        the rate limiter to its initial state.
        """
        self._request_timestamps.clear()
    
    def get_current_count(self) -> int:
        """
        Get the current number of requests in the sliding window.
        
        This method cleans up old timestamps and returns the count of
        requests within the current time window.
        
        Returns:
            int: Number of requests in the current sliding window.
        """
        current_time = time.time()
        cutoff_time = current_time - self.time_window
        
        # Remove timestamps outside the current time window
        while self._request_timestamps and self._request_timestamps[0] <= cutoff_time:
            self._request_timestamps.popleft()
        
        return len(self._request_timestamps)
    
    def time_until_next_request(self) -> float:
        """
        Calculate the time until the next request can be made.
        
        Returns:
            float: Time in seconds until a request can be made. Returns 0.0
                  if a request can be made immediately.
        """
        if len(self._request_timestamps) < self.max_requests:
            return 0.0
        
        # The oldest request timestamp + time_window is when we can make the next request
        oldest_timestamp = self._request_timestamps[0]
        next_available_time = oldest_timestamp + self.time_window
        current_time = time.time()
        
        return max(0.0, next_available_time - current_time)
    
    def __repr__(self) -> str:
        """
        Return a string representation of the RateLimiter.
        
        Returns:
            str: String representation showing the rate limiter configuration.
        """
        return (f"RateLimiter(max_requests={self.max_requests}, "
                f"time_window={self.time_window}, "
                f"current_count={self.get_current_count()})")
```

This implementation provides:

1. **Sliding Window Algorithm**: Uses a deque to efficiently track request timestamps and removes old ones as needed.

2. **Type Hints**: All methods include proper type annotations for parameters and return values.

3. **Comprehensive Docstrings**: Each method has detailed documentation explaining its purpose, parameters, and return values.

4. **Input Validation**: Constructor validates that parameters are positive integers.

5. **Additional Utility Methods**: 
   - `get_current_count()`: Returns current number of requests in the window
   - `time_until_next_request()`: Calculates when the next request can be made
   - `__repr__()`: Provides useful string representation

6. **Efficient Implementation**: Uses `collections.deque` for O(1) operations on both ends of the timestamp queue.

Example usage:
```python
# Create rate limiter: 5 requests per minute
limiter = RateLimiter(max_requests=5, time_window=60)

# Check if request is allowed
if limiter.is_allowed():
    print("Request processed")
else:
    wait_time = limiter.time_until_next_request()
    print(f"Rate limited. Try again in {wait_time:.2f} seconds")

# Check current status
print(f"Current requests in window: {limiter.get_current_count()}")

# Reset if needed
limiter.reset()
```

### Generating Code in Multiple Languages


In [7]:
# Claude can generate code in many languages
languages = ["Python", "JavaScript", "TypeScript", "Rust", "Go"]

task = "Write a function to reverse a string"

print("🌐 Same task in multiple languages:\n")

for lang in languages:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=300,
        messages=[{
            "role": "user",
            "content": f"{task}. Use {lang}. Just the code, no explanation."
        }]
    )
    
    print(f"📌 {lang}:")
    print(response.content[0].text)
    print("\n" + "-"*50 + "\n")


🌐 Same task in multiple languages:

📌 Python:
```python
def reverse_string(s):
    return s[::-1]
```

--------------------------------------------------

📌 JavaScript:
```javascript
function reverseString(str) {
    return str.split('').reverse().join('');
}
```

--------------------------------------------------

📌 TypeScript:
```typescript
function reverseString(str: string): string {
    return str.split('').reverse().join('');
}
```

--------------------------------------------------

📌 Rust:
```rust
fn reverse_string(s: &str) -> String {
    s.chars().rev().collect()
}
```

--------------------------------------------------

📌 Go:
```go
func reverseString(s string) string {
    runes := []rune(s)
    for i, j := 0, len(runes)-1; i < j; i, j = i+1, j-1 {
        runes[i], runes[j] = runes[j], runes[i]
    }
    return string(runes)
}
```

--------------------------------------------------



In [8]:
# Generating SQL queries from natural language
sql_prompt = """
I have a database with the following tables:
- users (id, name, email, created_at, subscription_tier)
- orders (id, user_id, total_amount, status, created_at)
- products (id, name, price, category)
- order_items (order_id, product_id, quantity)

Write SQL queries for:
1. Get top 10 customers by total spending
2. Find products that have never been ordered
3. Calculate monthly revenue for the past year
4. Find users who haven't ordered in the last 30 days
"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    system="You are a database expert. Write efficient, readable SQL queries.",
    messages=[{"role": "user", "content": sql_prompt}]
)

print("📊 Generated SQL Queries:")
display(Markdown(response.content[0].text))


📊 Generated SQL Queries:


Here are the SQL queries for each of your requirements:

## 1. Top 10 customers by total spending

```sql
SELECT 
    u.id,
    u.name,
    u.email,
    SUM(o.total_amount) AS total_spending
FROM users u
INNER JOIN orders o ON u.id = o.user_id
WHERE o.status != 'cancelled'  -- Assuming we want to exclude cancelled orders
GROUP BY u.id, u.name, u.email
ORDER BY total_spending DESC
LIMIT 10;
```

## 2. Products that have never been ordered

```sql
SELECT 
    p.id,
    p.name,
    p.price,
    p.category
FROM products p
LEFT JOIN order_items oi ON p.id = oi.product_id
WHERE oi.product_id IS NULL
ORDER BY p.name;
```

## 3. Monthly revenue for the past year

```sql
SELECT 
    DATE_FORMAT(o.created_at, '%Y-%m') AS month,
    SUM(o.total_amount) AS monthly_revenue,
    COUNT(o.id) AS order_count
FROM orders o
WHERE o.created_at >= DATE_SUB(CURDATE(), INTERVAL 1 YEAR)
    AND o.status != 'cancelled'  -- Exclude cancelled orders
GROUP BY DATE_FORMAT(o.created_at, '%Y-%m')
ORDER BY month DESC;
```

**Alternative for PostgreSQL:**
```sql
SELECT 
    TO_CHAR(o.created_at, 'YYYY-MM') AS month,
    SUM(o.total_amount) AS monthly_revenue,
    COUNT(o.id) AS order_count
FROM orders o
WHERE o.created_at >= CURRENT_DATE - INTERVAL '1 year'
    AND o.status != 'cancelled'
GROUP BY TO_CHAR(o.created_at, 'YYYY-MM')
ORDER BY month DESC;
```

## 4. Users who haven't ordered in the last 30 days

```sql
SELECT 
    u.id,
    u.name,
    u.email,
    u.subscription_tier,
    MAX(o.created_at) AS last_order_date
FROM users u
LEFT JOIN orders o ON u.id = o.user_id
GROUP BY u.id, u.name, u.email, u.subscription_tier
HAVING MAX(o.created_at) < DATE_SUB(CURDATE(), INTERVAL 30 DAY)
    OR MAX(o.created_at) IS NULL  -- Include users who have never ordered
ORDER BY last_order_date DESC;
```

**Alternative approach (more explicit):**
```sql
SELECT 
    u.id,
    u.name,
    u.email,
    u.subscription_tier
FROM users u
WHERE u.id NOT IN (
    SELECT DISTINCT o.user_id 
    FROM orders o 
    WHERE o.created_at >= DATE_SUB(CURDATE(), INTERVAL 30 DAY)
        AND o.user_id IS NOT NULL
)
ORDER BY u.name;
```

## Notes:

- **Query 1**: Excludes cancelled orders from spending calculation. Remove the WHERE clause if you want to include all orders.
- **Query 2**: Uses LEFT JOIN to find products with no matching order items.
- **Query 3**: Groups by year-month format. Adjust the date functions based on your database system (MySQL vs PostgreSQL vs others).
- **Query 4**: The first version shows the last order date, while the alternative simply lists users without recent orders. Choose based on your needs.

All queries include appropriate indexing considerations - make sure you have indexes on:
- `orders.user_id`
- `orders.created_at`
- `order_items.product_id`
- `users.id`

---
## 2. Explaining Existing Code

Claude can analyze and explain code at various levels of detail.


In [9]:
# Complex code to explain
complex_code = '''
def memoize(func):
    cache = {}
    def wrapper(*args, **kwargs):
        key = str(args) + str(sorted(kwargs.items()))
        if key not in cache:
            cache[key] = func(*args, **kwargs)
        return cache[key]
    return wrapper

@memoize
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)
'''

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{
        "role": "user",
        "content": f"""Explain this Python code in detail. Cover:
1. What each part does
2. How the decorator pattern works
3. Why memoization helps here
4. The time complexity improvement

```python
{complex_code}
```"""
    }]
)

print("📚 Code Explanation:")
display(Markdown(response.content[0].text))


📚 Code Explanation:


I'll break down this memoization example step by step.

## 1. What Each Part Does

### The Memoize Decorator
```python
def memoize(func):
    cache = {}  # Dictionary to store computed results
    
    def wrapper(*args, **kwargs):
        # Create a unique key from function arguments
        key = str(args) + str(sorted(kwargs.items()))
        
        # Check if result is already cached
        if key not in cache:
            cache[key] = func(*args, **kwargs)  # Compute and store
        
        return cache[key]  # Return cached result
    
    return wrapper
```

**Key components:**
- `cache = {}`: Persistent dictionary that survives between function calls
- `key = str(args) + str(sorted(kwargs.items()))`: Converts arguments into a hashable cache key
- Cache lookup: If key exists, return stored result; otherwise compute and store

### The Fibonacci Function
```python
@memoize
def fibonacci(n):
    if n < 2:
        return n  # Base cases: fib(0)=0, fib(1)=1
    return fibonacci(n - 1) + fibonacci(n - 2)  # Recursive formula
```

## 2. How the Decorator Pattern Works

The `@memoize` syntax is equivalent to:
```python
fibonacci = memoize(fibonacci)
```

**Step-by-step execution:**
1. `memoize(fibonacci)` is called, returning the `wrapper` function
2. `fibonacci` now points to `wrapper`, not the original function
3. When you call `fibonacci(5)`, you're actually calling `wrapper(5)`
4. `wrapper` checks the cache, then calls the original function if needed

**Visual representation:**
```
fibonacci(5) → wrapper(5) → [cache check] → original_fibonacci(5)
                ↓
            cache['(5,)'] = result
                ↓
            return result
```

## 3. Why Memoization Helps Here

### The Problem with Naive Fibonacci
Without memoization, `fibonacci(5)` creates this call tree:
```
fibonacci(5)
├── fibonacci(4)
│   ├── fibonacci(3)
│   │   ├── fibonacci(2)
│   │   │   ├── fibonacci(1) → 1
│   │   │   └── fibonacci(0) → 0
│   │   └── fibonacci(1) → 1
│   └── fibonacci(2)  # DUPLICATE WORK!
│       ├── fibonacci(1) → 1
│       └── fibonacci(0) → 0
└── fibonacci(3)      # MORE DUPLICATE WORK!
    ├── fibonacci(2)
    │   ├── fibonacci(1) → 1
    │   └── fibonacci(0) → 0
    └── fibonacci(1) → 1
```

### With Memoization
```python
# First call to fibonacci(5):
fibonacci(5) → cache miss → compute
├── fibonacci(4) → cache miss → compute
│   ├── fibonacci(3) → cache miss → compute
│   └── fibonacci(2) → cache hit! ← Returns cached value
└── fibonacci(3) → cache hit! ← Returns cached value

# Cache after fibonacci(5):
# {'(0,)': 0, '(1,)': 1, '(2,)': 1, '(3,)': 2, '(4,)': 3, '(5,)': 5}
```

## 4. Time Complexity Improvement

### Without Memoization: O(2^n)
- Each call spawns 2 recursive calls
- Total calls ≈ 2^n
- `fibonacci(30)` makes ~1 billion function calls!

### With Memoization: O(n)
- Each unique subproblem is solved exactly once
- We need to compute fibonacci(0) through fibonacci(n)
- Total: n+1 unique computations

**Dramatic improvement example:**
```python
import time

# Without memoization - takes several seconds
def fib_slow(

In [10]:
# Explaining code for different audiences
async_code = '''
async def fetch_all(urls):
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_one(session, url) for url in urls]
        return await asyncio.gather(*tasks)

async def fetch_one(session, url):
    async with session.get(url) as response:
        return await response.json()
'''

audiences = [
    ("beginner", "Explain to someone new to programming. Use simple analogies."),
    ("intermediate", "Explain to someone familiar with Python but new to async."),
    ("expert", "Focus on performance implications and best practices.")
]

print("👥 Same code, different audiences:\n")

for level, instruction in audiences:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": f"""{instruction}

```python
{async_code}
```"""
        }]
    )
    
    print(f"📖 {level.upper()} Explanation:")
    print(response.content[0].text[:500] + "..." if len(response.content[0].text) > 500 else response.content[0].text)
    print("\n" + "="*60 + "\n")


👥 Same code, different audiences:

📖 BEGINNER Explanation:
I'll explain this code using simple analogies that anyone can understand!

## What is Async Programming?

Think of **async programming** like being a waiter at a restaurant:

- **Regular programming** = You take one order, walk to the kitchen, wait for the food to be ready, bring it back, then take the next order
- **Async programming** = You take multiple orders, send them all to the kitchen at once, and while waiting, you can do other things. When each dish is ready, you deliver it

## Breakin...


📖 INTERMEDIATE Explanation:
I'll explain this async code by breaking it down and comparing it to what you'd normally do in regular Python.

## What's happening here?

This code fetches data from multiple URLs **concurrently** (at the same time) rather than one after another. Think of it like this:

**Regular Python (sequential):**
```python
def fetch_all_sync(urls):
    results = []
    for url in urls:
        response = requests.

In [11]:
# Code review and improvement suggestions
code_to_review = '''
def get_user_data(user_id):
    connection = mysql.connector.connect(host="localhost", user="root", password="password123", database="mydb")
    cursor = connection.cursor()
    cursor.execute("SELECT * FROM users WHERE id = " + str(user_id))
    result = cursor.fetchone()
    return result
'''

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1200,
    messages=[{
        "role": "user",
        "content": f"""Review this code for:
1. Security vulnerabilities
2. Best practices violations
3. Error handling issues
4. Provide an improved version

```python
{code_to_review}
```"""
    }]
)

print("🔍 Code Review:")
display(Markdown(response.content[0].text))


🔍 Code Review:


## Code Review

### 1. Security Vulnerabilities
- **SQL Injection**: Direct string concatenation creates vulnerability to SQL injection attacks
- **Hardcoded Credentials**: Database password exposed in source code
- **Overprivileged Access**: Using 'root' user for application connections

### 2. Best practices Violations
- **No Resource Management**: Database connections not properly closed
- **No Configuration Management**: Database settings hardcoded
- **Poor Error Handling**: No exception handling for database operations
- **Excessive Data Exposure**: Using `SELECT *` returns all user data unnecessarily

### 3. Error Handling Issues
- **No Exception Handling**: Database connection failures will crash the application
- **Silent Failures**: Function returns `None` without indicating why (user not found vs. error)
- **Resource Leaks**: Unclosed connections can exhaust connection pool

## Improved Version

Here are two improved implementations:

### Option 1: Basic Improvements
```python
import mysql.connector
from mysql.connector import Error
import os
from typing import Optional, Dict, Any
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def get_database_connection():
    """Create and return a database connection using environment variables."""
    try:
        connection = mysql.connector.connect(
            host=os.getenv('DB_HOST', 'localhost'),
            user=os.getenv('DB_USER'),
            password=os.getenv('DB_PASSWORD'),
            database=os.getenv('DB_NAME'),
            autocommit=True,
            connection_timeout=10
        )
        return connection
    except Error as e:
        logger.error(f"Database connection failed: {e}")
        raise

def get_user_data(user_id: int, fields: Optional[list] = None) -> Optional[Dict[str, Any]]:
    """
    Retrieve user data by ID with proper security and error handling.
    
    Args:
        user_id: The user ID to search for
        fields: Optional list of specific fields to retrieve
        
    Returns:
        Dictionary containing user data or None if not found
        
    Raises:
        ValueError: If user_id is invalid
        DatabaseError: If database operation fails
    """
    # Input validation
    if not isinstance(user_id, int) or user_id <= 0:
        raise ValueError("User ID must be a positive integer")
    
    # Define allowed fields to prevent data exposure
    allowed_fields = ['id', 'username', 'email', 'first_name', 'last_name', 'created_at']
    
    if fields:
        # Validate requested fields
        invalid_fields = set(fields) - set(allowed_fields)
        if invalid_fields:
            raise ValueError(f"Invalid fields requested: {invalid_fields}")
        select_fields = ', '.join(fields)
    else:
        select_fields = ', '.join(allowed_fields)
    
    connection = None
    cursor = None
    
    try:
        connection = get_database_connection()
        cursor = connection.cursor(dictionary=True)
        
        # Use parameterized query to prevent SQL injection
        query = f"SELECT {select_fields} FROM users WHERE id = %s"
        cursor.execute(query, (user_id,))
        
        result = cursor.fetchone()
        
        if result:
            logger.info(f"Successfully retrieved user data for ID: {user_id}")
            return result
        else:
            logger.info(f"No user found with ID: {user_id}")
            return None
            
    except Error as e:
        logger.error(f"Database error while fetching user {user_id}: {e}")
        raise RuntimeError(f"Failed to retrieve user data: {e}")
    
    finally:
        # Ensure resources are always cleaned up
        if cursor:
            cursor.close()
        if connection and connection.is_connected():
            connection.close()
```

### Option 2: Production-Ready with Connection Pooling
```python
import mysql.connector.pooling
from mysql.connector import Error
import os
from typing import Optional, Dict, Any, List
from contextlib import contextmanager
import logging
from functools import lru_cache

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class DatabaseManager:
    """Database manager with connection pooling and proper resource management."""
    
    def __init__(self):
        self._pool = None
        self._initialize_pool()
    
    def _initialize_pool(self):
        """Initialize the connection pool."""
        try:
            pool_config = {
                'pool_name': 'mypool',
                'pool_size': 10,
                'pool_reset_session': True,
                'host': os.getenv('DB_HOST', 'localhost'),
                'user': os.getenv('DB_USER'),
                'password': os.getenv('DB

---
## 3. Generating Comments and Documentation

Claude can add documentation at various levels: inline comments, docstrings, and full documentation.


In [12]:
# Code that needs documentation
undocumented_code = '''
class DataProcessor:
    def __init__(self, config):
        self.config = config
        self.data = None
        self._cache = {}
    
    def load(self, filepath):
        with open(filepath, 'r') as f:
            self.data = json.load(f)
        return self
    
    def transform(self, operations):
        if self.data is None:
            raise ValueError("No data loaded")
        
        result = self.data.copy()
        for op in operations:
            op_name = op['type']
            if op_name in self._cache:
                result = self._cache[op_name](result, op.get('params', {}))
            else:
                handler = getattr(self, f'_op_{op_name}', None)
                if handler:
                    self._cache[op_name] = handler
                    result = handler(result, op.get('params', {}))
        return result
    
    def _op_filter(self, data, params):
        field = params['field']
        value = params['value']
        return [d for d in data if d.get(field) == value]
    
    def _op_sort(self, data, params):
        return sorted(data, key=lambda x: x.get(params['field'], ''))
'''

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=2000,
    messages=[{
        "role": "user",
        "content": f"""Add comprehensive documentation to this code:
1. Class-level docstring (purpose, usage example)
2. Method docstrings (Google style with Args, Returns, Raises)
3. Inline comments for complex logic
4. Type hints

```python
{undocumented_code}
```

Return the fully documented code."""
    }]
)

print("📝 Documented Code:")
display(Markdown(response.content[0].text))


📝 Documented Code:


```python
import json
from typing import Dict, List, Any, Union, Callable, Optional


class DataProcessor:
    """A flexible data processing utility for JSON data with chainable operations.
    
    This class provides functionality to load JSON data from files and apply
    various transformation operations like filtering and sorting. Operations
    are cached for improved performance on repeated use.
    
    Attributes:
        config (Dict[str, Any]): Configuration dictionary for processor settings.
        data (Optional[List[Dict[str, Any]]]): The currently loaded data.
        
    Example:
        >>> config = {'max_cache_size': 100}
        >>> processor = DataProcessor(config)
        >>> result = (processor
        ...     .load('data.json')
        ...     .transform([
        ...         {'type': 'filter', 'params': {'field': 'status', 'value': 'active'}},
        ...         {'type': 'sort', 'params': {'field': 'name'}}
        ...     ]))
    """
    
    def __init__(self, config: Dict[str, Any]) -> None:
        """Initialize the DataProcessor with configuration.
        
        Args:
            config: Configuration dictionary containing processor settings.
                   Can include options like cache size limits, default behaviors, etc.
        """
        self.config = config
        self.data: Optional[List[Dict[str, Any]]] = None
        # Cache to store operation handlers for performance optimization
        self._cache: Dict[str, Callable] = {}

    def load(self, filepath: str) -> 'DataProcessor':
        """Load JSON data from a file.
        
        Args:
            filepath: Path to the JSON file to load. File should contain
                     a list of dictionaries or a single dictionary.
                     
        Returns:
            DataProcessor: Self for method chaining.
            
        Raises:
            FileNotFoundError: If the specified file doesn't exist.
            json.JSONDecodeError: If the file contains invalid JSON.
            IOError: If there are issues reading the file.
            
        Example:
            >>> processor = DataProcessor({}).load('users.json')
        """
        with open(filepath, 'r', encoding='utf-8') as f:
            self.data = json.load(f)
        return self

    def transform(self, operations: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Apply a series of transformation operations to the loaded data.
        
        Operations are applied sequentially in the order provided. Each operation
        is defined by a dictionary with 'type' and optional 'params' keys.
        Operation handlers are cached after first use for performance.
        
        Args:
            operations: List of operation dictionaries. Each operation should have:
                       - 'type': String identifying the operation (e.g., 'filter', 'sort')
                       - 'params': Optional dictionary of parameters for the operation
                       
        Returns:
            List[Dict[str, Any]]: The transformed data as a list of dictionaries.
            
        Raises:
            ValueError: If no data has been loaded before calling transform.
            AttributeError: If an operation type doesn't have a corresponding handler method.
            
        Example:
            >>> operations = [
            ...     {'type': 'filter', 'params': {'field': 'age', 'value': 25}},
            ...     {'type': 'sort', 'params': {'field': 'name'}}
            ... ]
            >>> result = processor.transform(operations)
        """
        if self.data is None:
            raise ValueError("No data loaded. Call load() method first.")

        # Work with a copy to avoid modifying original data
        result = self.data.copy()
        
        for op in operations:
            op_name = op['type']
            op_params = op.get('params', {})
            
            # Check if operation handler is already cached
            if op_name in self._cache:
                result = self._cache[op_name](result, op_params)
            else:
                # Dynamically get the operation handler method
                handler_name = f'_op_{op_name}'
                handler = getattr(self, handler_name, None)
                
                if handler:
                    # Cache the handler for future use
                    self._cache[op_name] = handler
                    result = handler(result, op_params)
                else:
                    raise AttributeError(f"Operation '{op_name}' not supported. "
                                       f"No method '{handler_name}' found.")
                    
        return result

    def _op_filter(self, data: List[Dict[str, Any]], 
                   params: Dict[str, Any]) -> List[Dict[str, Any]]:
        """Filter operation handler - filters data based on field equality.
        
        Args:
            data: List of dictionaries to filter.
            params: Parameters dictionary containing:
                   - 'field': The field name to filter on
                   - 'value': The value to match against
                   
        Returns:
            List[Dict[str, Any]]: Filtered list containing only items where
                                 the specified field equals the specified value.
                                 
        Example:
            >>> data = [{'name': 'John', 'age': 25}, {'name': 'Jane', 'age': 30}]
            >>> params = {'field': 'age', 'value': 25}
            >>> result = processor._op_filter(data, params)
            >>> # Returns [{'name': 'John', 'age': 25}]
        """
        field = params['field']
        value = params['value']
        # Use get() with None default to handle missing fields gracefully
        return [item for item in data if item.get(field) == value]

    def _op_sort(self, data: List[Dict[str, Any]], 
                 params: Dict[str, Any]) -> List[Dict[str, Any]]:
        """Sort operation handler - sorts data based on a specified field.
        
        Args:
            data: List of dictionaries to sort.
            params: Parameters dictionary containing:
                   - 'field': The field name to sort by
                   
        Returns:
            List[Dict[str, Any]]: New sorted list. Items missing the sort field
                                 will be sorted as empty strings (appear first).
                                 
        Example:
            >>> data = [{'name': 'John', 'age': 30}, {'name': 'Alice', 'age': 25}]
            >>> params = {'field': 'name'}
            >>> result = processor._op_sort(data, params)
            >>> # Returns [{'name': 'Alice', 'age': 25}, {'name': 'John', 'age': 30}]
        """
        field = params['field']
        # Use empty string as default for missing fields to ensure consistent sorting
        return sorted(data, key=lambda item: item.get(field, ''))
```

The comprehensive documentation includes:

1. **Class-level docstring**: Explains the purpose, provides attributes documentation, and includes a usage example
2. **Method docstrings**: Google style with Args, Returns, and Raises sections, plus examples where helpful
3. **Inline comments**: Explaining complex logic like caching, error handling, and data processing steps
4. **Type hints**: Complete type annotations for all parameters, return values, and class attributes
5. **Additional improvements**: Added encoding specification, better error messages, and more descriptive variable names in comments

In [13]:
# Generate different documentation styles
simple_function = '''
def calculate_discount(price, discount_percent, is_member=False):
    base_discount = price * (discount_percent / 100)
    if is_member:
        base_discount *= 1.1
    final_price = price - base_discount
    return max(0, final_price)
'''

doc_styles = [
    ("Google Style", "Use Google-style docstrings"),
    ("NumPy Style", "Use NumPy-style docstrings"),
    ("Sphinx Style", "Use Sphinx/reStructuredText style docstrings")
]

print("📚 Different Documentation Styles:\n")

for style_name, instruction in doc_styles:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": f"""{instruction} for this function:

```python
{simple_function}
```

Just show the function with the docstring, no explanation."""
        }]
    )
    
    print(f"📌 {style_name}:")
    print(response.content[0].text)
    print("\n" + "="*60 + "\n")


📚 Different Documentation Styles:

📌 Google Style:
```python
def calculate_discount(price, discount_percent, is_member=False):
    """Calculates the final price after applying a discount.
    
    Args:
        price (float): The original price of the item.
        discount_percent (float): The discount percentage to apply (0-100).
        is_member (bool, optional): Whether the customer is a member for additional
            discount. Defaults to False.
    
    Returns:
        float: The final price after discount, with a minimum value of 0.
    """
    base_discount = price * (discount_percent / 100)
    if is_member:
        base_discount *= 1.1
    final_price = price - base_discount
    return max(0, final_price)
```


📌 NumPy Style:
```python
def calculate_discount(price, discount_percent, is_member=False):
    """
    Calculate the final price after applying a discount.

    Parameters
    ----------
    price : float
        The original price of the item.
    discount_percen

In [14]:
# Generate README documentation
project_code = '''
# main.py
from dataclasses import dataclass
import requests

@dataclass
class WeatherData:
    temperature: float
    humidity: int
    description: str

class WeatherClient:
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://api.weather.com/v1"
    
    def get_current(self, city: str) -> WeatherData:
        response = requests.get(f"{self.base_url}/current", params={"city": city, "key": self.api_key})
        data = response.json()
        return WeatherData(data["temp"], data["humidity"], data["desc"])
    
    def get_forecast(self, city: str, days: int = 5) -> list[WeatherData]:
        response = requests.get(f"{self.base_url}/forecast", params={"city": city, "days": days, "key": self.api_key})
        return [WeatherData(d["temp"], d["humidity"], d["desc"]) for d in response.json()["days"]]
'''

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    messages=[{
        "role": "user",
        "content": f"""Generate a professional README.md for this Python library:

```python
{project_code}
```

Include:
1. Project title and description
2. Installation instructions
3. Quick start example
4. API reference
5. Contributing section
6. License placeholder"""
    }]
)

print("📄 Generated README:")
display(Markdown(response.content[0].text))


📄 Generated README:


# WeatherPy

A simple and intuitive Python library for fetching weather data. WeatherPy provides a clean interface to retrieve current weather conditions and forecasts for any city.

## Features

- 🌤️ Get current weather conditions
- 📅 Fetch weather forecasts up to multiple days
- 🎯 Simple, type-safe API with dataclasses
- 🔑 Easy API key authentication
- 📦 Lightweight with minimal dependencies

## Installation

Install WeatherPy using pip:

```bash
pip install weatherpy
```

Or install from source:

```bash
git clone https://github.com/yourusername/weatherpy.git
cd weatherpy
pip install -e .
```

## Requirements

- Python 3.9+
- requests

## Quick Start

```python
from weatherpy import WeatherClient

# Initialize the client with your API key
client = WeatherClient(api_key="your_api_key_here")

# Get current weather
current = client.get_current("New York")
print(f"Temperature: {current.temperature}°C")
print(f"Humidity: {current.humidity}%")
print(f"Conditions: {current.description}")

# Get 7-day forecast
forecast = client.get_forecast("London", days=7)
for day in forecast:
    print(f"Temp: {day.temperature}°C, {day.description}")
```

## API Reference

### WeatherData

A dataclass representing weather information.

**Attributes:**
- `temperature` (float): Temperature in Celsius
- `humidity` (int): Humidity percentage (0-100)
- `description` (str): Weather condition description

### WeatherClient

Main client class for interacting with the weather API.

#### `__init__(api_key: str)`

Initialize the WeatherClient with your API key.

**Parameters:**
- `api_key` (str): Your weather API key

**Example:**
```python
client = WeatherClient(api_key="your_api_key")
```

#### `get_current(city: str) -> WeatherData`

Get current weather conditions for a specified city.

**Parameters:**
- `city` (str): Name of the city

**Returns:**
- `WeatherData`: Current weather information

**Example:**
```python
weather = client.get_current("Paris")
print(f"Current temperature in Paris: {weather.temperature}°C")
```

#### `get_forecast(city: str, days: int = 5) -> list[WeatherData]`

Get weather forecast for a specified city.

**Parameters:**
- `city` (str): Name of the city
- `days` (int, optional): Number of forecast days (default: 5)

**Returns:**
- `list[WeatherData]`: List of weather forecasts

**Example:**
```python
forecast = client.get_forecast("Tokyo", days=3)
for i, day in enumerate(forecast):
    print(f"Day {i+1}: {day.temperature}°C, {day.description}")
```

## Error Handling

The library raises standard HTTP exceptions for API errors. It's recommended to handle these appropriately:

```python
import requests

try:
    weather = client.get_current("InvalidCity")
except requests.exceptions.HTTPError as e:
    print(f"HTTP error occurred: {e}")
except requests.exceptions.RequestException as e:
    print(f"Request failed: {e}")
```

## Configuration

### API Key

You can provide your API key in several ways:

1. **Direct initialization:**
   ```python
   client = WeatherClient(api_key="your_key")
   ```

2. **Environment variable:**
   ```python
   import os
   client = WeatherClient(api_key=os.getenv("WEATHER_API_KEY"))
   ```

## Examples

### Basic Usage
```python
from weatherpy import WeatherClient

client = WeatherClient("your_api_key")

# Current weather
current = client.get_current("San Francisco")
print(f"{current.description}, {current.temperature}°C")
```

### Multiple Cities
```python
cities = ["New York", "London", "Tokyo"]
for city in cities:
    weather = client.get_current(city)
    print(f"{city}: {weather.temperature}°C")
```

## Contributing

We welcome contributions! Please follow these steps:

1. Fork the repository
2. Create a feature branch (`git checkout -b feature/amazing-feature`)
3. Commit your changes (`git commit -m 'Add amazing feature'`)
4. Push to the branch (`git push origin feature/amazing-feature`)
5. Open a Pull Request

### Development Setup

```bash
# Clone the repo
git clone https://github.com/yourusername/weatherpy.git
cd weatherpy

# Install development dependencies
pip install -e ".[dev]"

# Run tests
python -m pytest

# Run linting
python -m flake8 weatherpy/
```

### Code Style

- Follow PEP 8
- Use type hints
- Add docstrings for public methods
- Write tests for new features

## License

This project is licensed under the MIT License - see the [LICENSE](LICENSE) file for details.

## Changelog

### v1.0.0
- Initial release
- Current weather and forecast functionality
- Type-safe WeatherData dataclass

---

**Note:** This library requires a valid API key from a weather service provider. Please ensure you comply with the terms of service of your chosen weather API provider.

---
## 4. Advanced Code Tasks

### Converting Between Languages


In [15]:
# Convert Python to JavaScript
python_code = '''
class ShoppingCart:
    def __init__(self):
        self.items = []
    
    def add_item(self, name, price, quantity=1):
        self.items.append({
            "name": name,
            "price": price,
            "quantity": quantity
        })
    
    def get_total(self):
        return sum(item["price"] * item["quantity"] for item in self.items)
    
    def apply_discount(self, percent):
        discount = self.get_total() * (percent / 100)
        return self.get_total() - discount
'''

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{
        "role": "user",
        "content": f"""Convert this Python class to modern JavaScript (ES6+):

```python
{python_code}
```

Use classes, arrow functions, and modern JS best practices."""
    }]
)

print("🔄 Python → JavaScript Conversion:")
display(Markdown(response.content[0].text))


🔄 Python → JavaScript Conversion:


Here's the Python class converted to modern JavaScript (ES6+):

```javascript
class ShoppingCart {
  constructor() {
    this.items = [];
  }

  addItem(name, price, quantity = 1) {
    this.items.push({
      name,
      price,
      quantity
    });
  }

  getTotal() {
    return this.items.reduce((total, item) => total + (item.price * item.quantity), 0);
  }

  applyDiscount(percent) {
    const total = this.getTotal();
    const discount = total * (percent / 100);
    return total - discount;
  }
}
```

Key differences and modern JS features used:

1. **Constructor syntax**: `constructor()` instead of `__init__()`
2. **Method naming**: camelCase (`addItem`, `getTotal`, `applyDiscount`) instead of snake_case
3. **Object shorthand**: `{ name, price, quantity }` instead of `{ name: name, price: price, quantity: quantity }`
4. **Arrow functions**: Used in the `reduce()` method for cleaner syntax
5. **Array.reduce()**: Modern alternative to Python's `sum()` with generator expression
6. **Default parameters**: `quantity = 1` works the same way in both languages
7. **Const declarations**: Used `const` for immutable values within methods

Example usage:
```javascript
const cart = new ShoppingCart();
cart.addItem("Apple", 1.50, 3);
cart.addItem("Banana", 0.75);
console.log(cart.getTotal()); // 5.25
console.log(cart.applyDiscount(10)); // 4.725
```

### Generating Unit Tests


In [16]:
# Generate tests for a function
function_to_test = '''
def validate_password(password: str) -> dict:
    """
    Validate a password against security rules.
    Returns a dict with 'valid' (bool) and 'errors' (list).
    """
    errors = []
    
    if len(password) < 8:
        errors.append("Password must be at least 8 characters")
    
    if not any(c.isupper() for c in password):
        errors.append("Password must contain at least one uppercase letter")
    
    if not any(c.islower() for c in password):
        errors.append("Password must contain at least one lowercase letter")
    
    if not any(c.isdigit() for c in password):
        errors.append("Password must contain at least one number")
    
    if not any(c in "!@#$%^&*" for c in password):
        errors.append("Password must contain at least one special character (!@#$%^&*)")
    
    return {"valid": len(errors) == 0, "errors": errors}
'''

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    messages=[{
        "role": "user",
        "content": f"""Generate comprehensive pytest unit tests for this function:

```python
{function_to_test}
```

Include:
1. Test valid passwords
2. Test each error condition
3. Edge cases
4. Parametrized tests where appropriate"""
    }]
)

print("🧪 Generated Unit Tests:")
display(Markdown(response.content[0].text))


🧪 Generated Unit Tests:


```python
import pytest
from typing import Dict, List

def validate_password(password: str) -> dict:
    """
    Validate a password against security rules.
    Returns a dict with 'valid' (bool) and 'errors' (list).
    """
    errors = []

    if len(password) < 8:
        errors.append("Password must be at least 8 characters")

    if not any(c.isupper() for c in password):
        errors.append("Password must contain at least one uppercase letter")

    if not any(c.islower() for c in password):
        errors.append("Password must contain at least one lowercase letter")

    if not any(c.isdigit() for c in password):
        errors.append("Password must contain at least one number")

    if not any(c in "!@#$%^&*" for c in password):
        errors.append("Password must contain at least one special character (!@#$%^&*)")

    return {"valid": len(errors) == 0, "errors": errors}


class TestValidatePassword:
    """Test suite for validate_password function."""

    def test_valid_passwords(self):
        """Test various valid password combinations."""
        valid_passwords = [
            "Password123!",
            "MySecure@Pass1",
            "ComplexP@ssw0rd",
            "Test123$",
            "Abcdefg1!",
            "P@ssw0rd123",
            "ValidPass1#",
            "StrongPwd2%",
            "SecureKey3^",
            "GoodPass4&",
            "TestPass5*"
        ]
        
        for password in valid_passwords:
            result = validate_password(password)
            assert result["valid"] is True, f"Password '{password}' should be valid"
            assert result["errors"] == [], f"Password '{password}' should have no errors"

    def test_minimum_length_requirement(self):
        """Test password length validation."""
        short_passwords = [
            "",           # empty string
            "A",          # 1 character
            "Ab1!",       # 4 characters
            "AbC1!23",    # 7 characters (one short)
        ]
        
        for password in short_passwords:
            result = validate_password(password)
            assert result["valid"] is False
            assert "Password must be at least 8 characters" in result["errors"]

    def test_uppercase_requirement(self):
        """Test uppercase letter validation."""
        no_uppercase_passwords = [
            "password123!",      # no uppercase
            "lowercase1@",       # no uppercase
            "test123#pass",      # no uppercase
            "mypassword2$"       # no uppercase
        ]
        
        for password in no_uppercase_passwords:
            result = validate_password(password)
            assert result["valid"] is False
            assert "Password must contain at least one uppercase letter" in result["errors"]

    def test_lowercase_requirement(self):
        """Test lowercase letter validation."""
        no_lowercase_passwords = [
            "PASSWORD123!",      # no lowercase
            "MYPASSWORD1@",      # no lowercase
            "TEST123#PASS",      # no lowercase
            "UPPERCASE2$"        # no lowercase
        ]
        
        for password in no_lowercase_passwords:
            result = validate_password(password)
            assert result["valid"] is False
            assert "Password must contain at least one lowercase letter" in result["errors"]

    def test_digit_requirement(self):
        """Test digit validation."""
        no_digit_passwords = [
            "Password!@#",       # no digits
            "MySecure@Pass",     # no digits
            "TestPassword!",     # no digits
            "NoNumbers@Here"     # no digits
        ]
        
        for password in no_digit_passwords:
            result = validate_password(password)
            assert result["valid"] is False
            assert "Password must contain at least one number" in result["errors"]

    def test_special_character_requirement(self):
        """Test special character validation."""
        no_special_passwords = [
            "Password123",       # no special chars
            "MySecurePass1",     # no special chars
            "TestPassword2",     # no special chars
            "NoSpecialChars3"    # no special chars
        ]
        
        for password in no_special_passwords:
            result = validate_password(password)
            assert result["valid"] is False
            assert "Password must contain at least one special character (!@#$%^&*)" in result["errors"]

    @pytest.mark.parametrize("special_char", ["!", "@", "#", "$", "%", "^", "&", "*"])
    def test_each_special_character_accepted(self, special_char):
        """Test that each allowed special character is accepted."""
        password = f"Password123{special_char}"
        result = validate_password(password)
        assert result["valid"] is True
        assert result["errors"] == []

    def test_invalid_special_characters_not_accepted(self):
        """Test that invalid special characters don't satisfy the requirement."""
        invalid_special_passwords = [
            "Password123+",      # + not in allowed set
            "Password123-",      # - not in allowed set
            "Password123=",      # = not in allowed set
            "Password123_",      # _ not in allowed set
            "Password123.",      # . not in allowed set
            "Password123?",      # ? not in allowed set
            "Password123,",      # , not in allowed set
            "Password123;",      # ; not in allowed set
        ]
        
        for password in invalid_special_passwords:
            result = validate_password(password)
            assert result["valid"] is False
            assert "Password must contain at least one special character (!@#$%^&*)" in result["errors"]

    def test_multiple_errors_combined(self):
        """Test passwords that fail multiple requirements."""
        # Test password with all possible errors
        result = validate_password("short")
        expected_errors = [
            "Password must be at least 8

### Refactoring Code


In [17]:
# Refactor messy code
messy_code = '''
def process(d):
    r = []
    for i in d:
        if i['type'] == 'A':
            x = i['value'] * 2
            if x > 100:
                x = 100
            r.append({'id': i['id'], 'result': x, 'category': 'doubled'})
        elif i['type'] == 'B':
            x = i['value'] + 10
            if x > 100:
                x = 100
            r.append({'id': i['id'], 'result': x, 'category': 'incremented'})
        elif i['type'] == 'C':
            x = i['value'] / 2
            if x < 0:
                x = 0
            r.append({'id': i['id'], 'result': x, 'category': 'halved'})
    return r
'''

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    messages=[{
        "role": "user",
        "content": f"""Refactor this code to be cleaner and more maintainable:

```python
{messy_code}
```

Apply:
1. Meaningful variable names
2. Extract helper functions
3. Use type hints
4. Add docstrings
5. Apply SOLID principles where appropriate
6. Use modern Python features (e.g., dataclasses, match statements)"""
    }]
)

print("✨ Refactored Code:")
display(Markdown(response.content[0].text))


✨ Refactored Code:


Here's a refactored version of the code that applies all the requested improvements:

```python
from dataclasses import dataclass
from typing import List, Dict, Any, Protocol


@dataclass
class ProcessingResult:
    """Represents the result of processing a data item."""
    id: str
    result: float
    category: str


class ProcessingStrategy(Protocol):
    """Protocol defining the interface for processing strategies."""
    
    def process(self, value: float) -> float:
        """Process a value according to the strategy."""
        ...
    
    @property
    def category(self) -> str:
        """Return the category name for this processing type."""
        ...


class DoubleStrategy:
    """Strategy for doubling values with upper bound constraint."""
    
    @property
    def category(self) -> str:
        return "doubled"
    
    def process(self, value: float) -> float:
        """Double the value, capped at 100."""
        return min(value * 2, 100)


class IncrementStrategy:
    """Strategy for incrementing values with upper bound constraint."""
    
    @property
    def category(self) -> str:
        return "incremented"
    
    def process(self, value: float) -> float:
        """Add 10 to the value, capped at 100."""
        return min(value + 10, 100)


class HalveStrategy:
    """Strategy for halving values with lower bound constraint."""
    
    @property
    def category(self) -> str:
        return "halved"
    
    def process(self, value: float) -> float:
        """Halve the value, with a floor of 0."""
        return max(value / 2, 0)


class DataProcessor:
    """Processor for handling different types of data items."""
    
    def __init__(self):
        self._strategies: Dict[str, ProcessingStrategy] = {
            'A': DoubleStrategy(),
            'B': IncrementStrategy(),
            'C': HalveStrategy()
        }
    
    def _process_single_item(self, item: Dict[str, Any]) -> ProcessingResult:
        """
        Process a single data item based on its type.
        
        Args:
            item: Dictionary containing 'id', 'type', and 'value' keys
            
        Returns:
            ProcessingResult containing the processed data
            
        Raises:
            KeyError: If the item type is not supported
            ValueError: If required keys are missing from the item
        """
        try:
            item_type = item['type']
            item_id = item['id']
            item_value = item['value']
        except KeyError as e:
            raise ValueError(f"Missing required key in item: {e}")
        
        if item_type not in self._strategies:
            raise KeyError(f"Unsupported item type: {item_type}")
        
        strategy = self._strategies[item_type]
        processed_value = strategy.process(item_value)
        
        return ProcessingResult(
            id=item_id,
            result=processed_value,
            category=strategy.category
        )
    
    def process_data(self, data_items: List[Dict[str, Any]]) -> List[ProcessingResult]:
        """
        Process a list of data items according to their types.
        
        Args:
            data_items: List of dictionaries, each containing 'id', 'type', and 'value'
            
        Returns:
            List of ProcessingResult objects containing processed data
            
        Example:
            >>> processor = DataProcessor()
            >>> data = [
            ...     {'id': '1', 'type': 'A', 'value': 60},
            ...     {'id': '2', 'type': 'B', 'value': 95},
            ...     {'id': '3', 'type': 'C', 'value': 20}
            ... ]
            >>> results = processor.process_data(data)
            >>> len(results)
            3
        """
        return [self._process_single_item(item) for item in data_items]


# Alternative implementation using match statement (Python 3.10+)
class ModernDataProcessor:
    """Modern implementation using match statements (requires Python 3.10+)."""
    
    def _process_single_item(self, item: Dict[str, Any]) -> ProcessingResult:
        """Process a single item using pattern matching."""
        try:
            item_type = item['type']
            item_id = item['id']
            item_value = item['value']
        except KeyError as e:
            raise ValueError(f"Missing required key in item: {e}")
        
        match item_type:
            case 'A':
                result = min(item_value * 2, 100)
                category = "doubled"
            case 'B':
                result = min(item_value + 10, 100)
                category = "incremented"
            case 'C':
                result = max(item_value / 2, 0)
                category = "halved"
            case _:
                raise KeyError(f"Unsupported item type: {item_type}")
        
        return ProcessingResult(id=item_id, result=result, category=category)
    
    def process_data(self, data_items: List[Dict[str, Any]]) -> List[ProcessingResult]:
        """Process data items using pattern matching."""
        return [self._process_single_item(item) for item in data_items]


# Usage example
if __name__ == "__main__":
    # Sample data
    sample_data = [
        {'id': '1', 'type': 'A', 'value': 60},
        {'id': '2', 'type': 'B', 'value': 95},
        {'id': '3', 'type': 'C', 'value': 20}
    ]
    
    # Using strategy pattern implementation
    processor = DataProcessor()
    results = processor.process_data(sample_data)
    
    for result in results:
        print(f"ID

### Debugging Assistance


In [18]:
# Debug code with an error
buggy_code = '''
def find_duplicates(items):
    """Find duplicate items in a list."""
    seen = {}
    duplicates = []
    
    for item in items:
        if item in seen:
            duplicates.append(item)
        seen[item] = True
    
    return duplicates

# Expected: [2, 3]
# Actual: [2, 2, 3]
result = find_duplicates([1, 2, 2, 2, 3, 3, 4])
'''

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{
        "role": "user",
        "content": f"""Debug this code. The expected output is [2, 3] but it returns [2, 2, 3]:

```python
{buggy_code}
```

1. Identify the bug
2. Explain why it happens
3. Provide the fixed code"""
    }]
)

print("🐛 Debug Analysis:")
display(Markdown(response.content[0].text))


🐛 Debug Analysis:


## 1. Identify the bug

The bug is that the function adds an item to the `duplicates` list **every time** it encounters a duplicate, not just the first time it identifies an item as a duplicate.

## 2. Explain why it happens

Let's trace through the execution with input `[1, 2, 2, 2, 3, 3, 4]`:

- `1`: Not in `seen` → add to `seen`
- `2`: Not in `seen` → add to `seen`  
- `2`: **In `seen`** → add to `duplicates` → `duplicates = [2]`
- `2`: **In `seen`** → add to `duplicates` → `duplicates = [2, 2]`
- `3`: Not in `seen` → add to `seen`
- `3`: **In `seen`** → add to `duplicates` → `duplicates = [2, 2, 3]`
- `4`: Not in `seen` → add to `seen`

The problem is that when we encounter the third occurrence of `2`, it's already in `seen`, so we add it to duplicates again. We need to track which items we've already identified as duplicates.

## 3. Provide the fixed code

```python
def find_duplicates(items):
    """Find duplicate items in a list."""
    seen = {}
    duplicates = []

    for item in items:
        if item in seen:
            # Only add to duplicates if this is the FIRST time we've seen it as a duplicate
            if seen[item] == 1:  # This means we've seen it exactly once before
                duplicates.append(item)
            seen[item] += 1
        else:
            seen[item] = 1

    return duplicates

# Test
result = find_duplicates([1, 2, 2, 2, 3, 3, 4])
print(result)  # Output: [2, 3]
```

**Alternative solution using a set to track added duplicates:**

```python
def find_duplicates(items):
    """Find duplicate items in a list."""
    seen = set()
    duplicates = []
    added_to_duplicates = set()

    for item in items:
        if item in seen and item not in added_to_duplicates:
            duplicates.append(item)
            added_to_duplicates.add(item)
        seen.add(item)

    return duplicates
```

Both solutions ensure that each duplicate item is only added to the result list once, regardless of how many times it appears in the input.

---
## Best Practices for Code Generation

### Tips for Better Results:

1. **Be Specific**: Include requirements, constraints, and expected behavior
2. **Provide Context**: Share relevant code, schemas, or examples
3. **Request Explanations**: Ask Claude to explain its choices
4. **Iterate**: Use follow-up prompts to refine the code
5. **Verify**: Always test generated code before using it

### Effective Prompt Patterns:

```python
# Pattern 1: Requirements first
"Create a function that:
- Takes X as input
- Returns Y
- Handles edge cases A, B, C
- Uses library Z"

# Pattern 2: Example-driven
"Write code that transforms this input:
[example input]
Into this output:
[example output]"

# Pattern 3: Style specification
"Using [language/framework], following [style guide], 
implement [feature] with [constraints]"
```


---
## Summary

In this segment, we covered:

1. **Generating Code**: From natural language descriptions to working code
2. **Explaining Code**: At different levels for different audiences
3. **Documentation**: Docstrings, comments, and README generation
4. **Advanced Tasks**: Language conversion, testing, refactoring, debugging

### Key Takeaways:
- Specific prompts yield better code
- Include requirements, constraints, and examples
- Use system prompts to set coding standards
- Always review and test generated code
- Claude can adapt to different styles and audiences

### Best Practices:
- Provide context (existing code, schemas, requirements)
- Request explanations to understand the code
- Use iterative refinement for complex tasks
- Specify language version and style guides
- Include edge cases in your prompts

---

## Course Wrap-Up

Congratulations! You've completed the Claude API for Python Developers course.

### What We Covered:

1. **Segment 1**: Generative AI fundamentals and API basics
2. **Segment 2**: Claude models, prompting, and summarization
3. **Segment 3**: Embeddings with Voyage AI for semantic search
4. **Segment 4**: Tools for structured outputs and agents
5. **Segment 5**: Code generation and explanation

### Next Steps:
- Build a project using what you've learned
- Explore the Anthropic documentation for advanced features
- Join the Anthropic community for updates and support

Happy coding with Claude! 🚀


<center>
     <img src="data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>